In [ ]:
import sys
import requests
sys.path.append(r'E:\wangzhilin\QuantumGalaxy')
sys.path.append(r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints')
from live_editor.models import bitable_works
import json
from QGI.feishu import base_fetch_func,fetch_tenant_token
from copy import deepcopy
bitapp_auth={'app_id':'cli_a1fc0a192db8100c','app_secret':'nNk1fT8h2eVi8B2ymIFangHTpYrOAJTD'}
def build_headers():
    
    token=fetch_tenant_token(bitapp_auth)
    #token='u-3atGsodiN9391_BlvmfeZh1kkwt5hlQxo0G0hgew2Ijc'
    headers = {
        'Authorization':
        'Bearer ' + token,  # your access token
        #'Content-Type': 'application/json; charset=utf-8'
    }
    return headers


In [ ]:
import logging
def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(
        r"E:\wangzhilin\QuantumGalaxy\opt\combine_check\log\log.log",
        'a',
        encoding='utf-8')
    fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter = logging.Formatter(
        #fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
        fmt='%(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger
logger=get_logger('combine_check')

In [ ]:
logger.warning('start')

#### 将2个（多个？）骨干多维表格中的外骨干节点和关系抽出，进行对照，排除冲突和错误

规则：
1. 表格内外部节点/边是否都有外部骨干名字，外部骨干名字（只有写了外部骨干才是外部节点）
2. 表格内外部节点和边是否意义对应（表格内有外部节点的话，则至少有1条外部边从内部节点或其他外部节点连向该外部节点），存在不对应的点和边则加入error，该表不联查 另，外部节点连向外部节点的边和节点都warning，不参与检查和报错

3. 满足以上两条规则的表格可以参与联合检查，每张表每张表排除自身error和warning后生成n组外边（主主体关系客体）及其外骨干名
对m张表的外边匹配相应外骨干的外边，如果匹配失败则加入该表的错误外边，成功的一对外边成为星图边界

4. 对各个粗骨干的内部节点命名进行唯一性检查，如果重复则是可能的外部节点
4. 最终生成匹配成功的星图边界表和其他表的错误报告（节点和边不对应error，外边匹配失败error，多层2边warning）
5. 画图（原本的内容+正确的外点外边，但是点和边有重复,解决冲突后用cypher可以录入）

In [ ]:
def get_mark(backbone_mark):
    if backbone_mark:
        mark=[]
        
        for record in backbone_mark:
            fields=record['fields']
            #print(fields)
            if fields['本骨干标记']:
                mark.append(fields['骨干名'])
        if mark:
            if len(mark)==1:
                return (True,mark[0])

            else:
                return (False,'找到复数勾选骨干：%s'%mark)
        else:
            return (False,'没有勾选骨干标记')
    else:
        return (False,'没有勾选本骨干')


In [ ]:
def get_files(fold_token):
    logger.info('开始检查文件，文件夹地址为　https://quantumgalaxy.feishu.cn/drive/folder/%s'%fold_token)
    r0=requests.get('http://82.156.248.152:81/fs/fold_meta?token=%s'%fold_token).json()
    
    r1=requests.get('http://82.156.248.152:81/fs/fold_children?fld_token=%s'%fold_token).json()
    if r0['code']==0 and r1['code']==0:
        files=r1['data']['files']
        data={}
        for file in files:
            if file['name'] not in ['电力储能骨干核心表Demo','xx骨干核心表模板及范例']:
            #if file['type']=='bitable' and '核心表' in file['name'] and file['name']!='xx骨干核心表Demo':
            #if True:
                data[file['name']]=file['token']
            else:
                logger.warning('排除该文件，类型:%s,名称:%s'%(file['type'],file['name']))
        num=len(data)
        
        logger.info('检查文件夹成功，共查到%d张多维表格参与联查，如下：'%num)
        for d in data.keys():
            logger.info(d)
        
        return {'code':0,'data':data}
    else:
        logger.info('检查文件夹失败，检查地址')
        return {'code':-1,'data':None}


In [ ]:
def external_check(internal_result,files):
    '''对所有表格中externallinks+backbonemark进行检查，其对应的
    external_link结构：  
    'external_links': [{'bitapp_id': 'recNx4LYVO',
    'source': 'reclwpDIsc',
    'target': 'recDC1O4Rf',
    'category': '父节点',
    'checked': True,
    'source_name': '精煤',
    'target_name': '动力煤',
    'external_backbone_name': '电力',
    'external_backbone': '未知'},
    
    A表 有externallink L1,其external_backbone为B，则B表必须有相同sourcenametargaetname的externallink L1，其externalbackbone为A。成功则L1 l2为正确边界(只判断L1在出报告更方便？），否则暴露L1为错误'''
    result=deepcopy(internal_result)
    for token,data in result.items():
        logger.critical('对标记为"%s"的表格的边界进行检查,该表共有%s条外边参与联调'%(data['backbone_mark'],len(data['external_links'])))
        logger.critical(data['backbone_mark'])
        data['error_external_links']=[]
        data['good_external_links']=[]
        #print(data['external_links'])
        for link in data['external_links']:
            linktext=link['source_name']+'--'+link['category']+'->'+link['target_name']
            other_token=link['external_backbone']
            #print(internal_result[other_token])
            #othertoken是在internalcheck时根据file表查到
            get=False
            if other_token in internal_result.keys():
                for other_link in internal_result[other_token]['external_links']:
                #当表格在files里，但权限有问题，该表的token不会出现在internalresult中，需要提示
                    otherlinktext=other_link['source_name']+'--'+other_link['category']+'->'+other_link['target_name']
                    if otherlinktext==linktext:
                        get=True
                        if other_link['external_backbone']==token:
                            try:
                                if link.get('target'):
                                    link['source']=other_link['source']
                                else:
                                    print(link)
                                    print(other_link)
                                    link['target']=other_link['target']
                                data['good_external_links'].append(link)
                                logger.info('星图边界识别成功，在"%s"骨干中查到对应边："%s"'%(link['external_backbone_name'],linktext))
                            except:
                               data['error_external_links'].append(link) 
                               logger.error('找到文本相同的星图边界但起终点有问题，可能是方向写反了或者外骨干标重了，如无法解决联系王之霖。边信息：%s'%linktext)
                        else:
                            
                            data['error_external_links'].append(link)
                            logger.error('星图边界识别到对应的边，但对应边的外骨干标记与本骨干不符，对应边的外骨干标记为："%s"，对应边为："%s"'%(link['external_backbone_name'],linktext))
                        break
            else:
                data['error_external_links'].append(link)
                logger.error('星图边界识别失败，该边："%s"所对应的"%s"骨干没有经过内部检查，这是对应骨干而非本边的问题'%(linktext,link['external_backbone_name']))
            if not get:
                data['error_external_links'].append(link)
                logger.error('星图边界识别失败，该边："%s"在对应"%s"骨干中没找到对应边界'%(linktext,link['external_backbone_name']))
        logger.critical('完成对标记为"%s"的表格的边界进行检查,该表共有%s条外边参与联调，通过边%s条，冲突边%s条\n'%(data['backbone_mark'],len(data['external_links']),len(data['good_external_links']),len(data['error_external_links'])))
        #print(data['backbone_mark'])
        data.pop('external_links')
    return result


In [ ]:
def internal_check(data1,files):
    '''对多维表格data进行内部检查，梳理出所有外部节点和其对应边，其中孤立外部节点加入error，多层外部节点加入warning，其他加入external,并把原data中其他数据分类到
    'internal_links':[],'external_links':[],'warning_links':[],'error_nodes':[],'internal_nodes':[],'external_nodes':[],
    加上骨干标记'''
    data=deepcopy(data1)
    result={'internal_links':[],'external_links':[],'warning_links':[],'error_links':[],'error_nodes':[],'internal_nodes':[],'external_nodes':[]}
    r,mark=get_mark(data['backbone_mark'])
    if r:
        logger.critical('本骨干标记：')
        logger.critical(mark)
        result['backbone_mark']=mark
    else:
        logger.error('检查到骨干定义表问题：%s'%mark)
        return None
    
    #result.update(data)
    for link in data['links']:
        if not link['category'] in ['原材料','储运','父节点','供应','同位','生产']:
            link['error']=True
            result['error_links'].append(link)
            logger.error('该边的关系有误：“%s”'%link['category'])
        for o in data['nodes']:
            if o['bitapp_id']==link['target']:
                link['target_name']=o['name']
            if o['bitapp_id']==link['source']:
                link['source_name']=o['name']
        if not (link.get('target_name') and link.get('source_name')):
            link['error']=True
            result['error_links'].append(link)
            logger.error('出现边没有找到对应的起/终点，可能是因为起终点没有勾选粗骨干权限。id:"%s",起点："%s" 终点:"%s"'%(link.get('bitapp_id'),link.get('source_name_ref','unknown'),link.get('target_name_ref','unknown')))
            
    for node in data['nodes']:
        if not node['more_info'].get('external_backbone'):
            result['internal_nodes'].append(node)
        else:
            
            id=node['bitapp_id']
            backbone=node['more_info'].get('external_backbone')

            if backbone:
                backbone=backbone[0]
            check=0
            if not files.get(backbone,None):
                '''该外部节点的‘所属外部骨干’字段与文件目录files比对结果，因此要求字段内骨干名与files名一致，不然认为没有找到外骨干'''
                '''涉及errornode的边移入errorlink'''
                logger.error('“%s”节点没有找到对应的外部骨干：“%s”'%(node['name'],backbone))
                #error
                result['error_nodes'].append(node)
                for link in data['links']:
                    if link['source']==id or link['target']=='id':
                        if link['source']==id :
                            other=link['target']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                    #link['source_name']=node['name']
                                    #link['target_name']=o['name']
                                    link['external_backbone_name']=backbone
                                    link['external_backbone']=files.get(backbone,'未知')
                        else:
                            other=link['source']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                    #link['source_name']=o['name']
                                    #link['target_name']=node['name']
                                    link['external_backbone_name']=backbone
                                    link['external_backbone']=files.get(backbone,'未知')
                        result['error_links'].append(link)
                        logger.error('因“%s”节点没有找到对应外部骨干，与其相连的边也出错,不参与联调：[%s]-(%s)->[%s]'%(node['name'],link['source_name'],link['category'],link['target_name']))
                        link['error']=True
            else:
                result['external_nodes'].append(node)

                for link in data['links']:
                    
                        if link['source']==id :
                            
                            #从data中除去该边，只留纯外边
                            other=link['target']
                            for o in data['nodes']:
                                if o['bitapp_id']==other :
                                        
                                        
                                        link['source_name']=node['name']
                                        link['target_name']=o['name']
                                        link['external_backbone_name']=backbone
                                        link['external_backbone']=files.get(backbone,'未知')
                                        if not (o['more_info'].get('external_backbone') or o['category']=='指标'):
                                            #说明另一点为内部
                                            link1=deepcopy(link)
                                            link1.pop('source')
                                            logger.info('检查到与“%s”骨干间的正确星图边界：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            result['external_links'].append(link1)
                                            check=max(check,2)
                                            
                                        else:
                                        
                                            if not link.get('checked'):
                                                result['warning_links'].append(link)
                                                #说明另一点为外部,该边加入warning
                                                logger.warning('检查到本骨干之外的,属于“%s”骨干的边，该边不参与冲突检查：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            check=max(check,1)
                                        continue
                            link['checked']=True

                                
                                
                        elif link['target']==id:
                            
                            other=link['source']
                            for o in data['nodes']:
                                if o['bitapp_id']==other:
                                        
                                        
                                        link['source_name']=o['name']
                                        link['target_name']=node['name']
                                        link['external_backbone_name']=backbone
                                        link['external_backbone']=files.get(backbone,'未知')
                                        if not (o['more_info'].get('external_backbone') or o['category']=='指标'):
                                            #说明另一点为内部,该边加入external
                                            link1=deepcopy(link)
                                            link1.pop('target')
                                            logger.info('检查到与“%s”骨干间的正确星图边界：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            result['external_links'].append(link1)
                                            check=max(check,2)
                                            
                                        else:
                                            
                                            if not link.get('checked'):
                                                
                                                result['warning_links'].append(link)
                                                #说明另一点为外部,该边加入warning
                                                logger.warning('检查到本骨干之外的,属于“%s”骨干的边，该边不参与冲突检查：[%s]-(%s)->[%s]'%(backbone,link['source_name'],link['category'],link['target_name']))
                                            check=max(check,1)
                                        continue
                            link['checked']=True
                        else:
                            #result['internal_links'].append(link)
                            pass
                            '''所有边均未触发则error 该外部节点为孤立，未连入当前骨干'''
                        
                if check==2:
                    #ok
                    pass
                elif check==1:
                    #warning
                    pass
                else:
                    logger.error('检查到孤立外骨干节点，该点为内部错误，不参与冲突检查：%s'%node['name'])
                    #error
                    result['error_nodes'].append(node)
                    pass
    for link in data['links']:
                if not link.get('checked'):
                    if not link.get('error'):
                        #没有check表示起终点均不是外部
                        result['internal_links'].append(link)
    return result
#data1=deepcopy(data)
#internal_check(data1)
            

In [ ]:
testfiles={'第一个高级权限设置和外骨干范例': 'bascnzsAjUXwr4gUkaIpqi8BBaf','第二个高级权限设置和外骨干范例':'bascn9HmmVcWhW7fYfQdpJZXtdh'}

In [ ]:
def find_common_elements(result,files):
    #检查所有表格里 internal_nodes 的name是否有重复，有的话给出表格token和节点name
    dictionary={}
    for k,v in result.items():
        names=[]
        #if v.get('good_external_nodes'):
        for item in v['internal_nodes']:
        #for item in v['good_external_nodes']:
            if item['category']!='企业':
                names.append(item['name'])
        dictionary[k]=names
    
    common_elements = []
    for key1, lst1 in dictionary.items():
        for key2, lst2 in dictionary.items():
            if key1 != key2:
                for element in lst1:
                    if element in lst2  and not ([element,(key1,key2)]  in common_elements or ([element,(key2,key1)]  in common_elements)):
                        common_elements.append([element,(key1,key2)])
    for i in common_elements:
        tokens=i.pop(1)
        names=({v:k for k,v in files.items()}[tokens[0]],{v:k for k,v in files.items()}[tokens[1]])
        i.append(names)

    return common_elements

In [ ]:
def work1(files):
    logger.info('start')

    
    
    raw_datas={}
    internals={}
    externals={}
    internal_result={}
    errorfile=[]
    #result={'raw':raw_datas,'internal':internals,'external':externals}
    for name,token in files.items():
        '''file:{name,token}'''

        #data=bitable_works(token,subgraph_name='粗骨干图')
        data=bitable_works(token)
        raw_datas[token]=data
        logger.critical('对 “%s”  多维表格进行内部检查'%name)
        if data.get('code'):
            logger.critical('对 “%s”  多维表格进行内部检查失败，错误信息：%s\n'%(name,data))
            errorfile.append(name)
            continue
        r=internal_check(data,files)
        if r:
            internal_result[token]=r
        logger.critical('完成对 “%s”  多维表格进行内部检查\n'%name)
    logger.critical('内部检查完成，以下表格出现问题不参与联调:%s\n'%errorfile)
    #for errorname in errorfile:
        #files.pop(errorname)
    return internal_result
def work2(internal_result,files):

    logger.critical('开始外部联调，参与表格为：')
    logger.critical(str(list(files.keys()))+'\n')
    
    result=external_check(internal_result,files)
    
    '''外部检查'''
        
    return result
    

In [ ]:
logger.info('start')

In [ ]:
fold='fldcn2VlYvt8Efb9PqZwO51GWud'
files=get_files(fold)
if files['code']==0:
        files=files['data']
files


In [ ]:
for n in files.keys():
    print(n)

In [ ]:
internal_result=work1(files)
internal_result



In [ ]:
result=work2(internal_result,files)



In [ ]:
dup=find_common_elements(result,files)
dup=[[d[0],d[1][0],d[1][1]] for d in dup]
dup
import pandas as pd
dupdf=pd.DataFrame(data=dup,columns=['节点名','所属骨干1','所属骨干2'])
dupdf.sort_values('节点名')

dupdf.to_csv(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\dupnames.csv')
dupdf

In [ ]:
def save_json(result):
    import json
    from datetime import datetime
    d=datetime.today().__format__("%Y%m%d")
    b = json.dumps(result)
    f2 = open(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\results\result-%s.json'%d, 'w')
    f2.write(b)
    f2.close()
save_json(result)

# result2cypher

对所有表格里的internalnodes internallinks+ good_external_links就ok？（goodexternallink 有两组,可能moreinfo还不相同,用merge？不相同时会有重边。需要提前融合外边

重名内部节点处理？ 公司名字有唯一性约束，其他没有。merge会对{}内的项进行完全匹配，内部重名和连线不会出错（需要联调时检查并warning）。只需要把公司做更新backbone和code
连边的时候需要对点明确骨干？bitappid！

node的数据项 backbone，配图（地址，后续做,backbonetoken+bitappid，后续每次刷库时可以用md5验证图片之后再下到本地？api没有md5，必须要下载吧），code,category,（x,y）

link的数据项 sourcename type targetname (产线地位)， 只用名字匹配？？

其他信息及其结构要改82的bitablework和81的飞书接口

In [17]:

vertexs=[]
links={}


nodedic={'产品':'Product','企业':'Company','指标':'Indicator','工艺':'Technique','需求':'Demand'}
linkdic={'原材料':'compose','供应':'supply','父节点':'parent','储运':'carry','生产':'produce','同位':'peer','反映':'measure'}
for file_token,data in result.items():
    print(data['backbone_mark'])
    print('内部节点:%s'%len(data['internal_nodes']))
    print('内部边:%s'%len(data['internal_links']))
    print('外部节点:%s'%len(data['external_nodes']))
    print('外部边:%s'%len(data['good_external_links']))
    for node in data['internal_nodes']:
        post_node=deepcopy(node)
        cate=post_node.pop('category')
        cate_new=nodedic.get(cate,cate)
        post_node['category']=cate_new
        more=post_node.pop('more_info')
        post_node['file_token']=file_token
        post_node['backbone']=data['backbone_mark']
        if more:
            post_node.update(more)
        vertexs.append(post_node)
    for link in data['internal_links']:
        post_link={}
        post_link['source']=link['source']
        post_link['target']=link['target']
        post_link['target_name']=link['target_name']
        post_link['source_name']=link['source_name']
        post_link['category']=linkdic.get(link['category'],link['category'])
        if link.get('more_info'):
            
            post_link.update(link['more_info'])
        key=link['source_name']+'--'+link['category']+'->'+link['target_name']
        if key not in links.keys():
            links[key]=post_link
        else:
            logger.warning("“%s”骨干内部重复的边:'%s'"%(data['backbone_mark'],key))
            links[key].update(post_link)
    for link in data['good_external_links']:
        #要找到其节点所在本骨干的recordid，不能直接用link里的id。。
        post_link={}
        post_link['source']=link['source']
        post_link['target']=link['target']
        post_link['target_name']=link['target_name']
        post_link['source_name']=link['source_name']
        post_link['category']=linkdic.get(link['category'],link['category'])
        if link.get('more_info'):
            post_link.update(link['more_info'])
        key=link['source_name']+'--'+link['category']+'->'+link['target_name']
        if key not in links.keys():
            links[key]=post_link
        else:
            #print("dup!:'%s'"%key)
            links[key].update(post_link)
post_result={'vertex':vertexs,'link':list(links.values())}
print(len(post_result['vertex']))
print(len(post_result['link']))
        


WARNING  “半导体”骨干内部重复的边:'华兴源创--生产->芯片封装测试设备'
WARNING  “半导体”骨干内部重复的边:'阿莱德--生产->射频器件'
WARNING  “半导体”骨干内部重复的边:'博创科技--生产->光模块'
WARNING  “半导体”骨干内部重复的边:'德科立--生产->光模块'
WARNING  “半导体”骨干内部重复的边:'铜箔--原材料->覆铜板'
WARNING  “半导体”骨干内部重复的边:'剑桥科技--生产->光模块'
WARNING  “半导体”骨干内部重复的边:'联特科技--生产->光模块'
WARNING  “半导体”骨干内部重复的边:'世嘉科技--生产->射频器件'
WARNING  “半导体”骨干内部重复的边:'中英科技--生产->覆铜板'
WARNING  “半导体”骨干内部重复的边:'光芯片--原材料->光模块'
WARNING  “半导体”骨干内部重复的边:'北方华创--生产->半导体薄膜沉积设备'
WARNING  “半导体”骨干内部重复的边:'北方华创--生产->刻蚀机'
WARNING  “半导体”骨干内部重复的边:'北方华创--生产->半导体清洗设备'
WARNING  “半导体”骨干内部重复的边:'澜起科技--生产->电源管理芯片，电源管理IC'
WARNING  “半导体”骨干内部重复的边:'澜起科技--生产->存储芯片'
WARNING  “半导体”骨干内部重复的边:'通信芯片--父节点->射频芯片'
WARNING  “光伏”骨干内部重复的边:'横店东磁--生产->HJT电池'
WARNING  “光伏”骨干内部重复的边:'横店东磁--生产->PERC电池'
WARNING  “光伏”骨干内部重复的边:'横店东磁--生产->TOPCon电池'
WARNING  “光伏”骨干内部重复的边:'珈伟新能--生产->光伏发电'
WARNING  “光伏”骨干内部重复的边:'晶澳科技--生产->光伏组件'
WARNING  “光伏”骨干内部重复的边:'天合光能--生产->光伏组件'
WARNING  “食品饮料”骨干内部重复的边:'科拓生物--生产->乳酸菌制品'
WARNING  “食品饮料”骨干内部重复的边:'线下零售渠道--父节点->农贸市场'
WARNING  “食品饮料”骨干内部重复的

地产
内部节点:385
内部边:467
外部节点:17
外部边:11
互联网云计算
内部节点:664
内部边:759
外部节点:19
外部边:3
畜牧业
内部节点:144
内部边:142
外部节点:17
外部边:18
航空航天
内部节点:157
内部边:169
外部节点:17
外部边:2
教育
内部节点:148
内部边:177
外部节点:0
外部边:0
汽车
内部节点:339
内部边:279
外部节点:6
外部边:5
半导体
内部节点:343
内部边:448
外部节点:30
外部边:18
光伏
内部节点:199
内部边:239
外部节点:6
外部边:2
钠离子电池
内部节点:94
内部边:105
外部节点:3
外部边:2
食品饮料
内部节点:463
内部边:461
外部节点:24
外部边:23
电影
内部节点:56
内部边:101
外部节点:0
外部边:0


WARNING  “电影”骨干内部重复的边:'洁面产品--父节点->氨基酸洗面奶'
WARNING  “电影”骨干内部重复的边:'酰基氨基酸盐--原材料->氨基酸洗面奶'
WARNING  “电影”骨干内部重复的边:'氢氧化钾--原材料->皂基洗面奶'
WARNING  “电影”骨干内部重复的边:'美妆产品（化妆品）--父节点->头发护理产品'
WARNING  “电影”骨干内部重复的边:'美妆产品（化妆品）--父节点->化妆品包材'
WARNING  “电影”骨干内部重复的边:'塑料--原材料->化妆品包材'
WARNING  “电影”骨干内部重复的边:'美妆产品（化妆品）--原材料->美容护肤需求'
WARNING  “电影”骨干内部重复的边:'保湿剂--父节点->化妆品级透明质酸（玻尿酸）'
WARNING  “电影”骨干内部重复的边:'华熙生物--生产->透明质酸'
WARNING  “电影”骨干内部重复的边:'华熙生物--生产->护肤品类'
WARNING  “电影”骨干内部重复的边:'乳化剂--原材料->护肤乳液'
WARNING  “种植”骨干内部重复的边:'永顺泰--生产->麦芽'
WARNING  “锂离子电池”骨干内部重复的边:'导电剂--父节点->碳纳米管导电浆料'
WARNING  “锂离子电池”骨干内部重复的边:'导电剂--父节点->石墨烯'
WARNING  “锂离子电池”骨干内部重复的边:'导电剂--父节点->导电炭黑'
WARNING  “锂离子电池”骨干内部重复的边:'风储一体化--原材料->电力'
WARNING  “锂离子电池”骨干内部重复的边:'隔膜添加剂--原材料->干法隔膜'
WARNING  “锂离子电池”骨干内部重复的边:'隔膜添加剂--原材料->湿法隔膜'
WARNING  “锂离子电池”骨干内部重复的边:'光储一体化--原材料->电力'
WARNING  “锂离子电池”骨干内部重复的边:'天奈科技--生产->碳纳米管导电浆料'
WARNING  “锂离子电池”骨干内部重复的边:'湘潭电化--生产->二氧化锰'
WARNING  “锂离子电池”骨干内部重复的边:'新宙邦--生产->锂电池电解液'
WARNING  “锂离子电池”骨干内部重复的边:'科达利--生产->锂电池结构件'
WARNING  “旅游”骨干内部重

种植
内部节点:214
内部边:197
外部节点:27
外部边:31
锂离子电池
内部节点:294
内部边:435
外部节点:3
外部边:3
消费电子
内部节点:43
内部边:44
外部节点:3
外部边:0
银行
内部节点:127
内部边:152
外部节点:3
外部边:4
旅游
内部节点:103
内部边:91
外部节点:7
外部边:4
旅游
内部节点:74
内部边:63
外部节点:0
外部边:0
钢铁
内部节点:156
内部边:149
外部节点:25
外部边:11
石油化工
内部节点:83
内部边:105
外部节点:21
外部边:10
石油天然气
内部节点:61
内部边:66
外部节点:5
外部边:5
陆运
内部节点:100
内部边:100
外部节点:10
外部边:5
工业制造
内部节点:262
内部边:275
外部节点:3
外部边:11
钠离子电池
内部节点:89
内部边:89
外部节点:0
外部边:0
电影
内部节点:74
内部边:79
外部节点:3
外部边:1
海运
内部节点:121
内部边:133
外部节点:9
外部边:9
电力储能
内部节点:333
内部边:410
外部节点:14
外部边:5
风电
内部节点:100
内部边:107
外部节点:2
外部边:1
纺织业
内部节点:94
内部边:70
外部节点:45
外部边:24
医药
内部节点:132
内部边:148
外部节点:0
外部边:0
煤炭
内部节点:77
内部边:58
外部节点:4
外部边:2
5529
6127


In [19]:
from QGI.neoapi import Neo4j
#neo4j+ssc://534ea9b7.databases.neo4j.io:7687
#QuantumGalaxy
#atlas
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"
#nova
#uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
#user = "neo4j"
#password = "QuantumGalaxy"
#neo=Neo4j(uri,user,password)
neo=Neo4j(uri,user,password)
#neo.simple_query('match (n) detach delete n')



In [25]:

def data2cypher(neo,data):
    cyphers=['match (n) detach delete n']
    from copy import deepcopy
    d=deepcopy(data)
    for v in d['vertex']:
        #merge 应该只有名字？和骨干？公司怎么办
        cate=v.pop('category')
        if cate=='Company' or cate=='Indicator':
            name=v.pop('name')
            backbone=v.pop('backbone')
            
            cypher='merge(n:%s {name:"%s"}) \
                on match set n.backbone=n.backbone+",%s" \
                on create set n.backbone="%s" \
                '%(cate,name,backbone,backbone)
            if v:
                other='with n set '
                
                for key,value in v.items():
                    if type(value)==str:
                        value=value.replace('"', '\\\"').replace("'", "\\\'")
                    other+='n.%s="%s",'%(key,value)
                other=other[:-1]
                cypher+=other
            cyphers.append(cypher)
        else:
            s='{'
            for key,value in v.items():
                s+=key+':'+'"%s",'%value
            s=s[:-1]
            s+='}'
            cypher='merge (n:%s %s)'%(cate,s)
            #print(cypher)
            cyphers.append(cypher)

            
            #cypher='merge (n:%s {name:"%s"}) set n.backbone=n.backbone+",%s"'%(cate,v['name'],v['backbone'])
            #print('try this instead:"%s"'%cypher)
            #try:neo.simple_query(cypher)
            #except:print('failed! pass')
    for l in d['link']:
        #link表中对name和bitapp同时匹配 match (n {name:"",bitapp_id:""})
        #边界是不是不对？eg 畜牧业 牛奶- 食品饮料-乳制品
        sourcename,source,targetname,target,cate=l.pop('source_name'),l.pop('source'),l.pop('target_name'),l.pop('target'),l.pop('category')
        more=''
        if l:
            for k,v in l.items():
                more+=k+':'+'"%s",'%v
            more=more[:-1]
        if cate=='produce':
            cypher='match (n {name:"%s"}),(n1 {name:"%s",bitapp_id:"%s"}) create (n)-[r:%s {%s}]->(n1)'%(sourcename,targetname,target,cate,more) 
        else:
            cypher='match (n {name:"%s",bitapp_id:"%s"}),(n1 {name:"%s",bitapp_id:"%s"}) create (n)-[r:%s {%s}]->(n1)'%(sourcename,source,targetname,target,cate,more)          
        '''                                                                                                                                                                                   -------------------------------------+'
        s=deepcopy(data['vertex'][e['source']])
        s_cate=s.pop('category')
        s1='{'
        for key,value in s.items():
            s1+=key+':'+'"%s",'%value
        s1=s1[:-1]
        s1+='}'
        
        t=deepcopy(data['vertex'][e['target']])
        t_cate=t.pop('category')
        s2='{'
        for key,value in t.items():
            s2+=key+':'+'"%s",'%value
        s2=s2[:-1]
        s2+='}'
        cypher='match (n:%s %s),(n1:%s %s) create (n)-[r:%s]->(n1)'%(s_cate,s1,t_cate,s2,e["rela"])
        '''
        cyphers.append(cypher)
        #print(cypher)
        #neo.simple_query(cypher)
    return cyphers
cyphers=data2cypher(neo,post_result)
print(len(cyphers))

11657


In [26]:
r=neo.muti_cypher(cyphers)
r

11657

In [ ]:
data=bitable_works('bascnzsAjUXwr4gUkaIpqi8BBaf',subgraph_name='粗骨干图')
data

In [ ]:
from copy import deepcopy
#data1=deepcopy(data)

In [ ]:
import time

In [ ]:
s=time.time()
time.sleep(5)
e=time.time()-s
e

In [ ]:
fold='fldcn2VlYvt8Efb9PqZwO51GWud'

In [ ]:

r=requests.get('http://82.156.248.152:81/fs/fold_meta?token=%s'%fold).json()
r


In [ ]:
r=requests.get('http://82.156.248.152:81/fs/fold_children?fld_token=%s'%fold).json()
r

In [ ]:
r.json()

In [ ]:
import simplejson
with open(r'E:\wangzhilin\QuantumGalaxy\opt\combine_check\results\result-20230404.json','r') as file:
    data=simplejson.load(file)

In [ ]:
web=result['bascnX3JNXBB9J428M3qHmJv1Me']
web

In [ ]:
json.dumps('键桥科技')

### 多维表格附件图片的相关杂务

将所有多维表格中的配图列按照appid/tblid/recid的路径保存二进制文件（需要格式？）到8383/static/img下，将路径保存到哪里？（neo，则需要维护所有文件的node表，包括node表token

多为表格存的是图片。。neo要存路径，联调需要对所有节点存路径

In [ ]:
from html import escape
import json
from urllib.parse import quote
import requests,json

In [ ]:
bitapp_auth={'app_id':'cli_a1fc0a192db8100c','app_secret':'nNk1fT8h2eVi8B2ymIFangHTpYrOAJTD'}
def build_headers():
    
    token=fetch_tenant_token(bitapp_auth)
    #token='u-3atGsodiN9391_BlvmfeZh1kkwt5hlQxo0G0hgew2Ijc'
    headers = {
        'Authorization':
        'Bearer ' + token,  # your access token
        #'Content-Type': 'application/json; charset=utf-8'
    }
    return headers

In [ ]:
def download(url):
    url=url
    headers=build_headers()
    payload = ''
    #print(headers)
    file_token='bascnzsAjUXwr4gUkaIpqi8BBaf'
    table_token='tbl0Ko7cxJaGwUnh'
    record_id="recV96vgmB"
    save_path=os.path.join(r'E:\wangzhilin\QuantumGalaxy\pics',file_token)
    save_path=os.path.json(r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints\pic\static',file_token)
    #save_file=os.path.join(save_path,record_id)
    res=  requests.request("GET", url, headers=headers, data=payload,stream=True)
    if res.headers['Content-Type'] in ['image/gif', 'image/png', 'image/jpeg', 'image/bmp', 'image/webp']:
        post=res.headers['Content-Type'].split('/')[1]
        #post=res.headers['Content-Disposition'].split('.')[-1]
        name=record_id+'.'+post
        save_file=os.path.join(save_path,name)
        #return res.headers
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        with  open(save_file, "wb") as f:
            # 下载文件大小
            total_size = int(res.headers["content-length"])
            content_size = 0
            for content in res.iter_content(chunk_size=1024):
                f.write(content)
                content_size += len(content)

        data = {"recv_size": content_size, "total_size": total_size}
        if content_size == total_size:
            return ("接收成功")
        return ("接收失败")
    else:return('wrong type, got "%s"'%res.headers['Content-Type'])

r=download(r"https://open.feishu.cn/open-apis/drive/v1/medias/boxcnyh7dn98QVYTIGQZI4hPCOc/download?extra=%7B%22bitablePerm%22%3A%7B%22tableId%22%3A%22tbl0Ko7cxJaGwUnh%22%2C%22rev%22%3A77%7D%7D")
r

In [ ]:
def upload(file):
    UPDATE_PACKAGE_PATH = '/tools/package/'
    out = os.path.join(UPDATE_PACKAGE_PATH, params.get("filename"))
    if not os.path.isfile(out):
        return None
    # 文件大小
    fsize = os.path.getsize(out)

    def send_file_fp():
        store_path = out
        send_size = 0
        with open(store_path, "rb") as target_file:
            while 1:
                data = target_file.read(2 * 1024 * 1024)  # 每次读取2M
                if not data:
                    break
                yield data
    #content_type='multipart/form-data; boundary=something'
    content_type='image/'+name.split('.')[1]
    response = Response(send_file_fp(), )
    response.headers["Content-disposition"] = 'attachment; filename={}'.format(params.get("filename"))
    response.headers["Content-length"] = fsize
    return response


In [ ]:
def find_file(token,id):
    import os
    root_dir=r'E:\wangzhilin\QuantumGalaxy\server_new\blueprints\pic\static'
    file_dir=os.path.join(root_dir,token)
    files=os.listdir(file_dir)
    file_name=None
    for file in files:
        name=os.path.splitext(file)
        if name[0]==id:
            file_name=name[0]+name[1]
    return  file_dir,file_name
   
file_token='bascnzsAjUXwr4gUkaIpqi8BBaf'
table_token='tbl0Ko7cxJaGwUnh'
record_id="recV96vgmB"
r=find_file(file_token,record_id)

In [ ]:
from flask import Response
import os
def test():
    file_token='bascnzsAjUXwr4gUkaIpqi8BBaf'
    table_token='tbl0Ko7cxJaGwUnh'
    record_id="recV96vgmB"
    file_dir,file_name=find_file(file_token,record_id)
    if file_name:
        out=os.path.join(file_dir,file_name)
        fsize = os.path.getsize(out)

        def send_file_fp():
            store_path = out
            send_size = 0
            with open(store_path, "rb") as target_file:
                while 1:
                    data = target_file.read(2 * 1024 * 1024)  # 每次读取2M
                    if not data:
                        break
                    yield data
        #content_type='multipart/form-data; boundary=something'
        content_type='image/'+out.split('.')[1]
        response = Response(send_file_fp(),content_type=content_type )
        response.headers["Content-disposition"] = 'attachment; filename={}'.format(file_name)
        response.headers["Content-length"] = fsize
        return response
r=test()
r

In [ ]:
r.headers

In [ ]:
r.content

In [ ]:
imagebyte=r.content

In [ ]:
import cv2
import numpy as np

def read2byte(path):
    '''
    图片转二进制
    path：图片路径
    byte_data：二进制数据
    '''
    with open(path,"rb") as f:
        byte_data = f.read()
    return byte_data

def byte2numpy(byte_data):
    '''
    byte转numpy矩阵/cv格式
    byte_data：二进制数据
    image : numpy矩阵/cv格式图片
    '''
    image = np.asarray(bytearray(byte_data), dtype="uint8")
    image = cv2.imdecode(image, cv2.IMREAD_COLOR)
    return image


整合部分
对各个表格data进行整合（排除其中error的部分？）